In [0]:
from delta.table import DeltaTable
from pyspark.sql import functions as F
import datetime as dt

try:
    arrival_date = dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date = dt.date.today().strftime("%Y-%m-%d")
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "travel_bookings"
try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "default"

# read the booking data from bronze layer
book = spark.table(f"{catalog}.bronze.booking_inc").where(F.col("business_date") == F.to_date(F.lit(arrival_date)))

customer_dim = spark.sql(f"""SELECT customer_sk, customer_id FROM {catalog}.{schema}.customer_dim WHERE is_current = true""")

# Load current customer dimension records with surrogate keys
# Joins booking data with customer dimension to get surrogate
book_enriched = (book.alias(b)
                 .join(customer_dim.alias(c), F.col(b.customer_id) == F.col(c.customer_id), "left")
                 .withColumn("customer_sk", F.col(c.customer_sk))
)

# Select relevant columns for fact table aggregation
# Includes booking_type, customer keys, business_date, and financial metrics
book_enriched_sel = book_enriched.select(
    F.col("booking_type"),
    F.col("b.customer_id").alias("customer_id"),
    F.col("customer_sk"),
    F.col("business_date"),
    F.col("amount"),
    F.col("discount"),
    F.col("quantity")
)

# Aggregate booking data to daily grain by booking_type, customer, and date
# Calculates total amount (after discount) and total quantity
# Daily grain provides idempotent processing and business-friendly aggregation
book_agg = book_enriched_sel.groupBy(
    F.col("booking_type"),
    F.col("customer_sk"),
    F.col("customer_id"),
    F.col("business_date")
).agg(
    F.sum(F.col("amount") - F.col("discount")).alias("total_amount"),
    F.sum(F.col("quantity")).alias("total_quantity")
)

fact_full_name = f"{catalog}.{schema}.booking_fact"

# Fact table Merge Operation
book_agg.createOrReplaceTempView("src")
if not spark.catalog.tableExists(fact_full_name):
    a = book_agg.limit(0)
    a.write.format("delta").mode("overwrite").saveAsTable(fact_full_name)

spark.sql(f"""
          MERGE INTO {fact_full_name} t
          USING src s
          ON t.booking_type = s.booking_type 
          AND t.customer_sk = s.customer_sk 
          AND t.business_date = s.business_date
          WHEN MATCHED THEN UPDATE SET 
          t.total_amount = s.total_amount,
          t.total_quantity = s.total_quantity,
          t.customer_id = s.customer_id
          WHEN NOT MATCHED THEN INSERT *
          """)
print("Fact build complete")


